# integrity_checks

> Integrity check display with pass/fail indicators

In [ ]:
#| default_exp components.integrity_checks

In [ ]:
#| export
from typing import Any

from fasthtml.common import Div, Span, H3

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.card import card, card_body
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, shadow_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons via factory
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_transcript_verify.html_ids import VerifyHtmlIds
from cjm_transcript_verify.models import VerificationResult

## Check Row Component

In [ ]:
#| export
def _render_check_row(
    passed:bool,  # Whether the check passed
    label:str,  # Check description
    detail:str="",  # Optional detail (e.g., "246/246")
) -> Any:  # Check row element
    """Render an integrity check row with pass/fail icon."""
    if passed:
        icon = lucide_icon("circle-check", size=5, cls=str(text_dui.success))
    else:
        icon = lucide_icon("triangle-alert", size=5, cls=str(text_dui.warning))
    
    detail_span = Span(
        detail,
        cls=combine_classes(text_dui.base_content.opacity(60), font_size.sm)
    ) if detail else None
    
    return Div(
        icon,
        Span(label, cls=str(font_size.sm)),
        detail_span,
        cls=combine_classes(flex_display, items.center, gap(2), p.y(1))
    )

## Integrity Section

In [ ]:
#| export
def render_integrity_checks(
    result:VerificationResult,  # Verification result with integrity data
) -> Any:  # Integrity checks card
    """Render the structure integrity checks section."""
    # Compute expected counts for display
    expected_next = max(0, result.segment_count - 1)
    
    header = Div(
        lucide_icon("shield-check", size=5),
        H3("Structure Integrity", cls=combine_classes(font_size.lg, font_weight.semibold)),
        cls=combine_classes(flex_display, items.center, gap(2), m.b(3))
    )
    
    checks = Div(
        # STARTS_WITH check
        _render_check_row(
            passed=result.has_starts_with and result.starts_with_count == 1,
            label="STARTS_WITH edge",
            detail=str(result.starts_with_count)
        ),
        
        # NEXT chain check
        _render_check_row(
            passed=result.next_chain_complete,
            label="NEXT chain",
            detail=f"{result.next_count}/{expected_next}"
        ),
        
        # PART_OF check
        _render_check_row(
            passed=result.part_of_complete,
            label="PART_OF edges",
            detail=f"{result.part_of_count}/{result.segment_count}"
        ),
        
        # Timing check
        _render_check_row(
            passed=result.all_have_timing,
            label="All segments have timing",
            detail="" if result.all_have_timing else f"{result.segments_missing_timing} missing"
        ),
        
        # Sources check
        _render_check_row(
            passed=result.all_have_sources,
            label="All segments have sources",
            detail="" if result.all_have_sources else f"{result.segments_missing_sources} missing"
        ),
        
        cls=combine_classes(flex_display, flex_direction.col)
    )
    
    return Div(
        Div(
            header,
            checks,
            cls=str(card_body)
        ),
        id=VerifyHtmlIds.INTEGRITY_SECTION,
        cls=combine_classes(card, bg_dui.base_100, shadow.sm, shadow_dui.primary)
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()